# Entity Processor
> 
Pipeline:
- **add** raw JSON (or Entity) â†’ compose context  
- **expand** via ContextProcessor  
- **ingest** into GraphManager  
- **index** in a trivial dict for fast lookup

In [ ]:
#| default_exp core.processor

In [ ]:
#| export
from __future__ import annotations
from typing import Dict, Any, List
from collections import Counter
import uuid

from pyld import jsonld
from pyld.jsonld import JsonLdError

from cogitarelink.core.debug import get_logger
from cogitarelink.core.context import ContextProcessor
from cogitarelink.core.entity import Entity
from cogitarelink.core.graph import GraphManager

log = get_logger("processor")

## processor

In [ ]:
#| export
class EntityProcessor:
    def __init__(self,
                 ctx_proc: ContextProcessor | None = None,
                 graph: GraphManager | None = None):
        self.ctx   = ctx_proc or ContextProcessor()
        self.graph = graph or GraphManager()
        self.index: Dict[str, Entity] = {}
        self.vocab_counter: Counter[str] = Counter()
        # Track relationships between entities for better navigation
        self.parent_children: Dict[str, List[str]] = {}
        self.child_parent: Dict[str, str] = {}
        # Track credential subjects for VC handling
        self.credential_subjects: Dict[str, Dict[str, Entity]] = {}
        # Track @graph containers and their entries
        self.graph_containers: Dict[str, List[str]] = {}

    # ------------------------------------------------------------------
    def add(self, data: Dict[str, Any], vocab=["schema"]) -> Entity:
        """
        Add a new entity (and all child entities) to the processor
        
        Parameters
        ----------
        data: Raw content for the entity
        vocab: List of vocabulary prefixes to use for context
        
        Returns
        -------
        The created Entity instance
        """
        # Handle @graph arrays specially
        if "@graph" in data and isinstance(data["@graph"], list):
            return self._process_graph_document(data, vocab)
        
        # Special handling for credential subjects if using VC vocabulary
        if "vc" in vocab and isinstance(data, dict):
            # Check for credentialSubject field which needs special treatment
            if "credentialSubject" in data and isinstance(data["credentialSubject"], dict):
                # Create a copy of the data without the credentialSubject
                # This prevents the VC entity from containing the subject directly
                vc_data = data.copy()
                subject_data = vc_data.pop("credentialSubject")
                
                # Create the main VC entity 
                ent = Entity(vocab=vocab, content=vc_data)
                self._ingest_entity(ent)
                
                # Generate an ID for the subject if it doesn't have one
                subject_id = subject_data.get("@id", subject_data.get("id"))
                if not subject_id:
                    subject_id = f"urn:uuid:{uuid.uuid4()}"
                    subject_data["@id"] = subject_id
                
                # Create the subject entity with only schema vocabulary to avoid conflicts
                subject_entity = Entity(vocab=["schema"], content=subject_data)
                self._ingest_entity(subject_entity, parent_id=ent.id)
                
                # Add a reference back to the subject in the VC entity
                ent.content["credentialSubject"] = {"@id": subject_id}
                
                # Store in credential_subjects map for easy retrieval
                if ent.id not in self.credential_subjects:
                    self.credential_subjects[ent.id] = {}
                self.credential_subjects[ent.id][subject_id] = subject_entity
                
                return ent
        
        # Standard entity creation
        ent = Entity(vocab=vocab, content=data)
        self._ingest_entity(ent)
        return ent
    
    def _process_graph_document(self, data: Dict[str, Any], vocab: List[str]) -> Entity:
        """
        Process a document with @graph array
        
        Parameters
        ----------
        data: Document with @graph array
        vocab: List of vocabulary prefixes
        
        Returns
        -------
        Container Entity for the @graph document
        """
        # Create the container entity
        container = Entity(vocab=vocab, content=data)
        graph_entries = []
        
        # Process each entry in the @graph array
        for graph_entry in data.get("@graph", []):
            if not isinstance(graph_entry, dict):
                continue
                
            # Generate a unique ID if not provided
            if "@id" not in graph_entry and "id" not in graph_entry:
                graph_entry["@id"] = f"urn:graph:{uuid.uuid4()}"
                
            # Create an entity for this graph entry
            # If entry doesn't have context, it inherits from container
            if "@context" not in graph_entry:
                # Clone the context from container
                entry_with_context = {**graph_entry}
                # We'll apply context during entity creation via vocab
            else:
                entry_with_context = graph_entry
                
            entry_entity = Entity(vocab=vocab, content=entry_with_context)
            self._ingest_entity(entry_entity, parent_id=container.id)
            
            # Track this as a graph entry
            graph_entries.append(entry_entity.id)
            
        # Track graph entries in the container
        self.graph_containers[container.id] = graph_entries
        
        # Process the container entity itself
        self._ingest_entity(container)
        
        return container

    def _ingest_entity(self, ent: Entity, parent_id: str = None):
        """
        Process entity and its children recursively
        
        Parameters
        ----------
        ent: Entity to ingest
        parent_id: Optional ID of parent entity
        """
        # Expand and normalize for graph store
        try:
            expanded = self.ctx.expand(ent.as_json)
            nquads = self.ctx.normalize(expanded)
        except jsonld.JsonLdError as e:
            # Handle protected term redefinition error by trying a different strategy
            if "protected term redefinition" in str(e):
                log.warning(f"Protected term conflict for entity {ent.id}: {e}")
                # Try with a graph partition strategy
                try:
                    # Wrap the context in a way that isolates it
                    context_copy = ent.as_json.copy()
                    if isinstance(context_copy.get("@context"), dict):
                        context_copy["@context"] = [{"@vocab": "http://schema.org/"}, context_copy["@context"]]
                    expanded = self.ctx.expand(context_copy)
                    nquads = self.ctx.normalize(expanded)
                except Exception as e2:
                    log.error(f"Failed to resolve protected term conflict: {e2}")
                    raise
            else:
                raise
        
        # Store in graph with proper relationship tracking
        if parent_id:
            # Track the parent-child relationship
            if parent_id not in self.parent_children:
                self.parent_children[parent_id] = []
            self.parent_children[parent_id].append(ent.id)
            self.child_parent[ent.id] = parent_id
            
            # Use dedicated graph ID based on parent
            graph_id = f"{parent_id}#child_{ent.id}"
            self.graph.ingest_nquads(nquads, graph_id)
        else:
            # Root entity goes in default graph
            self.graph.ingest_nquads(nquads)
        
        # Index the entity for fast lookup
        self.index[ent.sha256] = ent
        self.vocab_counter.update(ent.vocab)
        
        # Process all child entities
        for child in ent.children:
            self._ingest_entity(child, parent_id=ent.id)
            
    def get_by_id(self, entity_id: str) -> Entity | None:
        """Retrieve entity by ID (not hash)"""
        for entity in self.index.values():
            if entity.id == entity_id:
                return entity
        return None
    
    def get_children(self, entity_id: str) -> List[Entity]:
        """Get all child entities for a given entity ID"""
        child_ids = self.parent_children.get(entity_id, [])
        return [self.get_by_id(child_id) for child_id in child_ids if self.get_by_id(child_id)]
    
    def get_parent(self, entity_id: str) -> Entity | None:
        """Get parent entity for a given entity ID"""
        parent_id = self.child_parent.get(entity_id)
        if parent_id:
            return self.get_by_id(parent_id)
        return None
    
    def get_credential_subjects(self, vc_id: str) -> List[Entity]:
        """Get credential subject entities for a given VC entity ID"""
        subjects = self.credential_subjects.get(vc_id, {})
        return list(subjects.values())
    
    def get_graph_entries(self, container_id: str) -> List[Entity]:
        """Get all graph entries for a @graph container entity"""
        entry_ids = self.graph_containers.get(container_id, [])
        return [self.get_by_id(entry_id) for entry_id in entry_ids if self.get_by_id(entry_id)]

## quick test

In [ ]:
#| hide
#| eval: false
from fastcore.test import test_eq, test_ne
ep = EntityProcessor()
e  = ep.add({"name": "Alan"})
test_eq(ep.index[e.sha256].content["name"], "Alan")
test_ne(ep.graph.size(), 0)

In [ ]:
#| hide
#| eval: false
# Test credential subject handling
from fastcore.test import test_eq, test_ne
import uuid

# Create a VC with credential subject
vc_data = {
    "@type": "VerifiableCredential",
    "issuer": "https://example.org/issuer",
    "credentialSubject": {
        "name": "Alice",
        "@type": "Person"
    }
}

# Use the processor to add the VC
ep = EntityProcessor()
vc = ep.add(vc_data, vocab=["vc", "schema"])

# Test that credential subject is stored separately but linked
test_eq(isinstance(vc.content.get("credentialSubject"), dict), True, "VC should have reference to subject")
test_eq("@id" in vc.content.get("credentialSubject", {}), True, "Subject reference should have @id")

# Test that credential subject was processed correctly
subjects = ep.get_credential_subjects(vc.id)
test_eq(len(subjects), 1, "Should have one credential subject")
test_eq(subjects[0].content["name"], "Alice", "Subject should have correct name")

# Test parent-child relationship
children = ep.get_children(vc.id)
test_eq(len(children), 1, "VC should have one child entity")
test_eq(children[0].content["name"], "Alice", "Child should be the credential subject")

# Test parent lookup
parent = ep.get_parent(children[0].id)
test_eq(parent.id, vc.id, "Parent should be the VC")

In [ ]:
#| hide
#| eval: false
# Test @graph handling
from fastcore.test import test_eq, test_ne
import uuid

# Create a test document with @graph array
graph_doc = {
    "@context": {
        "@vocab": "http://schema.org/"
    },
    "@graph": [
        {
            "@type": "Person",
            "name": "Alice"
        },
        {
            "@type": "Person",
            "name": "Bob",
            "@id": "urn:graph:bob"
        }
    ]
}

# Process with EntityProcessor
ep = EntityProcessor()
container = ep._process_graph_document(graph_doc, vocab=["schema"])

# Verify container was created
test_ne(container, None, "Container should exist")
test_eq(container.content.get("@graph"), graph_doc["@graph"], "Container should have original @graph array")

# Test graph entries
graph_entries = ep.get_graph_entries(container.id)
test_eq(len(graph_entries), 2, "Should have 2 graph entries")

# Check that one has our specified ID
bob_entry = None
for entry in graph_entries:
    if entry.id == "urn:graph:bob":
        bob_entry = entry
        break
        
test_ne(bob_entry, None, "Should find entry with ID 'urn:graph:bob'")
test_eq(bob_entry.content.get("name"), "Bob", "Entry should have correct content")

# Check that the other has a generated UUID-based ID
alice_entry = None
for entry in graph_entries:
    if entry.id != "urn:graph:bob":
        alice_entry = entry
        break
        
test_ne(alice_entry, None, "Should find entry with generated ID")
test_eq(alice_entry.content.get("name"), "Alice", "Entry should have correct content")
test_eq(alice_entry.id.startswith("urn:graph:"), True, "Generated ID should use urn:graph: scheme")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()